## Étape 4 — Embeddings et indexation

In [23]:
import json
import numpy as np
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

with open("../data/chunks.json", encoding="utf-8") as f:
    chunks = json.load(f)
print(len(chunks), "chunks chargés")

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

69 chunks chargés


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2678.37it/s]


## Q8 : dimension et volume

In [24]:
vecteur = model.encode("Combien de jours de télétravail par semaine ?")
dim = len(vecteur)

print("Dimension d'un vecteur :", dim)
print("Nombres à stocker :", f"{len(chunks)} chunks x {dim} = {len(chunks) * dim:,}")
print("Mémoire (float32) :", round(len(chunks) * dim * 4 / 1e6, 2), "Mo")

Dimension d'un vecteur : 384
Nombres à stocker : 69 chunks x 384 = 26,496
Mémoire (float32) : 0.11 Mo


## Q9 : matrice de similarité

In [25]:
phrases = [
    "Comment réinitialiser mon mot de passe ?",         
    "J'ai oublié mes identifiants de connexion.",       
    "Combien de jours de congés puis-je poser ?",       
    "Le restaurant d'entreprise ouvre à 11 h 45.",      
    "Je voudrais connaître mon solde de vacances.",     
]

emb = model.encode(phrases, normalize_embeddings=True)
similarite = emb @ emb.T       

etiquettes = [f"{i+1}" for i in range(len(phrases))]
pd.DataFrame(np.round(similarite, 2), index=etiquettes, columns=etiquettes)

,1,2,3,4,5
1,1.00,0.36,0.08,-0.02,0.08
2,0.36,1.00,-0.06,0.11,0.15
3,0.08,-0.06,1.00,0.11,0.53
4,-0.02,0.11,0.11,1.00,0.17
5,0.08,0.15,0.53,0.17,1.00


## Q10 : indexation dans Chroma

In [30]:
client = chromadb.PersistentClient(path="../base_vectorielle")
collection = client.get_or_create_collection(
    name="altea",
    metadata={"hnsw:space": "cosine"},     
)

textes = [c["texte"] for c in chunks]
embeddings = model.encode(textes, normalize_embeddings=True, batch_size=32, show_progress_bar=True)

collection.upsert(
    ids=[c["id"] for c in chunks],
    documents=textes,
    embeddings=embeddings.tolist(),
    metadatas=[{"source": c["source"], "page": c["page"], "service": c["service"]} for c in chunks],
)

print("Chunks produits :", len(chunks))
print("Chunks indexés  :", collection.count())
assert collection.count() == len(chunks)

Batches: 100%|██████████| 3/3 [00:00<00:00,  3.18it/s]

Chunks produits : 69
Chunks indexés  : 69


## Q11 : Premiere recherche 

In [31]:
def chercher(question: str, k: int = 3):
    q = model.encode([question], normalize_embeddings=True)
    res = collection.query(query_embeddings=q.tolist(), n_results=k)
    print(f"\n❓ {question}")
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        print(f"  score {1 - dist:.3f} | {meta['service']:7} | {meta['source']} p.{meta['page']}")
        print("     ", doc[:150].replace("\n", " "), "...")

chercher("Combien de jours de télétravail par semaine ?")
chercher("J'ai oublié mon mot de passe, que faire ?")
chercher("À quelle heure ouvre le restaurant d'entreprise ?")


❓ Combien de jours de télétravail par semaine ?
  score 0.709 | RH      | teletravail.pdf p.1
      bles. Les alternants et les stagiaires peuvent télétravailler 1 jour par semaine, avec l'accord de leur tuteur. 2. Nombre de jours autorisés Chaque sa ...
  score 0.615 | RH      | conges_absences.pdf p.1
      de congés payés par mois de travail effectif, soit 30 jours ouvrables (5 semaines) pour une année complète. La période d'acquisition court du 1er juin ...
  score 0.575 | RH      | conges_absences.pdf p.1
      oins 2 semaines à l'avance pour une absence de 1 à 3 jours, et au moins 1 mois à l'avance pour une absence de plus d'une semaine. ...

❓ J'ai oublié mon mot de passe, que faire ?
  score 0.698 | IT      | mot_de_passe.txt p.1
      MOT DE PASSE OUBLIÉ OU EXPIRÉ Altéa Services — Support IT — Référence IT-003  Vous avez oublié votre mot de passe, ou votre compte est bloqué ? Voici  ...
  score 0.591 | IT      | charte_informatique.pdf p.1
      utilisateur dispose d'un compt

## Q12 Que faudra-t-il faire le jour où un document est modifié ?

In [32]:
def reindexer_document(source: str, nouveaux_chunks: list[dict]):
    collection.delete(where={"source": source})       
    collection.upsert(                                  
        ids=[c["id"] for c in nouveaux_chunks],
        documents=[c["texte"] for c in nouveaux_chunks],
        embeddings=model.encode([c["texte"] for c in nouveaux_chunks], normalize_embeddings=True).tolist(),
        metadatas=[{"source": c["source"], "page": c["page"], "service": c["service"]} for c in nouveaux_chunks],
    )




In [ ]:
source = "teletravail.pdf"
anciens = [c for c in chunks if c["source"] == source]
print(len(anciens), "chunks pour", source)

avant = collection.count()
reindexer_document(source, anciens[:-1])     
print("Après modification :", avant, "->", collection.count())   

reindexer_document(source, anciens)          
print("Après restauration :", collection.count())              

7 chunks pour teletravail.pdf
Après modification : 69 -> 68
Après restauration : 69
